**Look Under the Hood: DICOM Metadata Explorer**

In the first notebook [here](https://www.kaggle.com/code/h17ann/1-rsna-knee-simple-starting-point-eda), we looked at the CSV files and the general dataset structure.

Now let's open the DICOM headers and see what information is stored there.

I will keep this notebook focused on metadata only. The actual MRI pixels will be explored in the next notebook.

We will look at:

- manufacturer
- patient sex and body part
- scanning sequence
- acquisition type
- magnetic field strength
- image size
- slice thickness
- spacing between slices
- pixel spacing

## 1. Imports

In [ ]:
import kagglehub
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pydicom as pdc
from pydicom.multival import MultiValue

from tqdm.auto import tqdm

## 2. Data

In [ ]:
DATA_DIR = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")
# Download data if it is not already attached
if not DATA_DIR.exists():
    DATA_DIR = Path(
        kagglehub.competition_download(
            "rsna-knee-abnormality-detection"
        )
    )
TRAIN_DIR = DATA_DIR / "train_series"

train_series = pd.read_csv(DATA_DIR / "train_series.csv")

display(train_series.head())
print("Studies:", train_series["StudyInstanceUID"].nunique())
print("Series :", train_series["SeriesInstanceUID"].nunique())

The image files are organized as:

`StudyInstanceUID / SeriesInstanceUID / *.dcm`

For this notebook I will use a sample of series so the metadata extraction stays reasonably fast.

## 3. Pick a sample of series

In [ ]:
N_SERIES = 300

sample_series = train_series.sample(
    n=min(N_SERIES, len(train_series)),
    random_state=42
)

sample_series.head()

In [ ]:
dicom_paths = []

for row in sample_series.itertuples(index=False):
    series_dir = (
        TRAIN_DIR
        / str(row.StudyInstanceUID)
        / str(row.SeriesInstanceUID)
    )

    dicom_paths.extend(series_dir.glob("*.dcm"))

print("DICOM files found:", len(dicom_paths))

## 4. Read one DICOM header

In [ ]:
dcm_path = dicom_paths[0]

ds = pdc.dcmread(
    dcm_path,
    stop_before_pixels=True
)

print(ds)

`stop_before_pixels=True` means that we read the DICOM header without loading the image pixels.

That is enough for this notebook and makes the reading faster.

## 5. Metadata we want to extract

In [ ]:
elements_list = [
    "study_uid",
    "series_uid",
    "modality",
    "manufacturer",
    "patient_sex",
    "body_part",
    "scanning_sequence",
    "acquisition_type",
    "magnetic_field_strength",
    "spacing_between_slices",
    "slice_thickness",
    "rows",
    "columns",
    "pixel_spacing"
]

tags_list = [
    (0x0020, 0x000D),  # StudyInstanceUID
    (0x0020, 0x000E),  # SeriesInstanceUID
    (0x0008, 0x0060),  # Modality
    (0x0008, 0x0070),  # Manufacturer
    (0x0010, 0x0040),  # PatientSex
    (0x0018, 0x0015),  # BodyPartExamined
    (0x0018, 0x0020),  # ScanningSequence
    (0x0018, 0x0023),  # MRAcquisitionType
    (0x0018, 0x0087),  # MagneticFieldStrength
    (0x0018, 0x0088),  # SpacingBetweenSlices
    (0x0018, 0x0050),  # SliceThickness
    (0x0028, 0x0010),  # Rows
    (0x0028, 0x0011),  # Columns
    (0x0028, 0x0030)   # PixelSpacing
]

## 6. Extract the metadata

In [ ]:
metadata_ = {
    "path": [],
    **{name: [] for name in elements_list}
}

for dcm_path in tqdm(dicom_paths):

    ds = pdc.dcmread(
        dcm_path,
        stop_before_pixels=True
    )

    metadata_["path"].append(str(dcm_path))

    for name, tag in zip(elements_list, tags_list):

        content = ds.get(tag)

        if content is not None:
            value = content.value

            if isinstance(value, MultiValue):

                if name == "pixel_spacing":
                    value = [float(v) for v in value]
                else:
                    value = "\\".join(str(v) for v in value)

        else:
            value = None

        metadata_[name].append(value)

metadata_df = pd.DataFrame(metadata_)

display(metadata_df.head())
print("Shape:", metadata_df.shape)

Each row of `metadata_df` corresponds to one DICOM slice.

A lot of metadata are repeated across all slices of the same MRI series.

## 7. Save it once

Before saving, multi-valued DICOM fields are converted to regular Python values.

- `PixelSpacing` becomes a normal list of two floats.
- Other `MultiValue` fields are stored as simple text.

This keeps the Parquet export happy.

In [ ]:
metadata_df.to_parquet(
    "/kaggle/working/dicom_metadata_sample.parquet",
    index=False
)

This is useful because extracting DICOM metadata can take some time.  
We can save the result and reuse it later instead of reading every DICOM file again.

## 8. Slice level vs series level

In [ ]:
print("DICOM slices :", len(metadata_df))
print("Unique series:", metadata_df["series_uid"].nunique())
print("Unique studies:", metadata_df["study_uid"].nunique())

For acquisition metadata, I prefer to count each MRI series once.

Otherwise, a series with 60 slices would have twice the weight of a series with 30 slices.

In [ ]:
series_df = metadata_df.drop_duplicates("series_uid").copy()

print("Series-level table:", series_df.shape)

## 9. Missing metadata

In [ ]:
missing = (
    series_df
    .isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

display(missing.round(2).to_frame("missing_%"))

In [ ]:
plt.figure(figsize=(10, 6))

ax = sns.barplot(
    x=missing.values,
    y=missing.index
)

for container in ax.containers:
    ax.bar_label(container, fmt="%.1f%%", padding=3)
    
plt.xlabel("Missing values (%)")
plt.ylabel("")
plt.title("Missing DICOM metadata")
plt.show()

Some DICOM tags are optional, so missing values are expected for some fields. If a missing field is required for a specific analysis, the corresponding series may need to be excluded from that analysis.

## 10. Modality

In [ ]:
plt.figure(figsize=(6, 4))

sns.countplot(
    data=series_df,
    x="modality",
    stat="percent"
)

plt.ylabel("Percent")
plt.title("Modality")
plt.show()

## 11. Manufacturer

In [ ]:
plt.figure(figsize=(10, 5))

ax = sns.countplot(
    data=series_df,
    x="manufacturer",
    stat="percent"
)

plt.xticks(rotation=45)
plt.ylabel("Percent")
plt.title("MRI manufacturer")
plt.show()

## 12. Patient sex

In [ ]:
study_df = metadata_df.drop_duplicates("study_uid").copy()

plt.figure(figsize=(6, 4))

sns.countplot(
    data=study_df,
    x="patient_sex",
    stat="percent"
)

plt.ylabel("Percent")
plt.title("Patient sex")
plt.show()

For patient sex I use one row per study instead of one row per slice.

## 13. Body part

In [ ]:
series_df["body_part"].unique()

In [ ]:
plt.figure(figsize=(8, 4))

sns.countplot(
    data=series_df,
    x="body_part",
    stat="percent"
)

plt.xticks(rotation=45)
plt.ylabel("Percent")
plt.title("Body part examined")
plt.show()

## 14. Scanning sequence

In [ ]:
series_df["scanning_sequence"] = series_df["scanning_sequence"].astype(str)

print("Unique values:", series_df["scanning_sequence"].nunique())

display(
    series_df["scanning_sequence"]
    .value_counts()
    .head(15)
    .to_frame("series")
)

In [ ]:
top_sequences = (
    series_df["scanning_sequence"]
    .value_counts()
    .head(12)
    .index
)

plot_df = series_df[
    series_df["scanning_sequence"].isin(top_sequences)
]

plt.figure(figsize=(11, 6))

sns.countplot(
    data=plot_df,
    y="scanning_sequence",
    order=top_sequences
)

plt.xlabel("Number of series")
plt.ylabel("")
plt.title("Most common scanning sequences")
plt.show()

## 15. Acquisition type

In [ ]:
plt.figure(figsize=(7, 4))

sns.countplot(
    data=series_df,
    x="acquisition_type",
    stat="percent"
)

plt.ylabel("Percent")
plt.title("MR acquisition type")
plt.show()

## 16. Magnetic field strength

In [ ]:
print(series_df["magnetic_field_strength"].value_counts(dropna=False))

In [ ]:
plt.figure(figsize=(7, 4))

sns.countplot(
    data=series_df,
    x="magnetic_field_strength",
    stat="percent"
)

plt.xlabel("Magnetic field strength (T)")
plt.ylabel("Percent")
plt.title("Magnetic field strength")
plt.show()

`MagneticFieldStrength` is expressed in Tesla.

## 17. Image matrix

In [ ]:
print("Unique row values   :", series_df["rows"].nunique())
print("Unique column values:", series_df["columns"].nunique())

In [ ]:
plt.figure(figsize=(10, 5))

sns.histplot(
    data=series_df,
    x="rows",
    bins=20,
    stat="percent"
)

plt.xlabel("Rows")
plt.ylabel("Percent")
plt.title("Image rows")
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))

sns.histplot(
    data=series_df,
    x="columns",
    bins=20,
    stat="percent"
)

plt.xlabel("Columns")
plt.ylabel("Percent")
plt.title("Image columns")
plt.show()

`Rows` and `Columns` are the image dimensions in pixels.

They do not tell us the physical size of the pixels.

## 18. Slice thickness

In [ ]:
series_df["slice_thickness"] = pd.to_numeric(
    series_df["slice_thickness"],
    errors="coerce"
)

display(series_df["slice_thickness"].describe().to_frame())

In [ ]:
plt.figure(figsize=(10, 5))

sns.histplot(
    data=series_df,
    x="slice_thickness",
    bins=20
)

plt.xlabel("Slice thickness (mm)")
plt.title("Slice thickness")
plt.show()

In [ ]:
plt.figure(figsize=(10, 2.5))

sns.boxplot(
    data=series_df,
    x="slice_thickness"
)

plt.xlabel("Slice thickness (mm)")
plt.show()

## 19. Spacing between slices

In [ ]:
series_df["spacing_between_slices"] = pd.to_numeric(
    series_df["spacing_between_slices"],
    errors="coerce"
)

display(
    series_df["spacing_between_slices"]
    .describe()
    .to_frame()
)

In [ ]:
plt.figure(figsize=(10, 5))

sns.histplot(
    data=series_df,
    x="spacing_between_slices",
    bins=20
)

plt.xlabel("Spacing between slices (mm)")
plt.title("Spacing between slices")
plt.show()

`SliceThickness` and `SpacingBetweenSlices` are not exactly the same thing.

Also, `SpacingBetweenSlices` is not always present in the DICOM header.

## 20. Pixel spacing

`PixelSpacing` contains two values:

`[row spacing, column spacing]`

The unit is millimeters per pixel.

In [ ]:
series_df["pixel_spacing_row"] = series_df["pixel_spacing"].apply(
    lambda x: float(x[0]) if x is not None else np.nan
)

series_df["pixel_spacing_col"] = series_df["pixel_spacing"].apply(
    lambda x: float(x[1]) if x is not None else np.nan
)

display(
    series_df[
        ["pixel_spacing_row", "pixel_spacing_col"]
    ].describe()
)

In [ ]:
plt.figure(figsize=(10, 5))

sns.histplot(
    data=series_df,
    x="pixel_spacing_row",
    bins=25
)

plt.xlabel("Row pixel spacing (mm/pixel)")
plt.title("Pixel spacing")
plt.show()

## 21. Are the in-plane pixels isotropic?

In [ ]:
spacing_df = series_df.dropna(
    subset=["pixel_spacing_row", "pixel_spacing_col"]
).copy()

same_spacing = np.isclose(
    spacing_df["pixel_spacing_row"],
    spacing_df["pixel_spacing_col"]
)

print(
    "Same row and column spacing:",
    f"{same_spacing.mean() * 100:.2f}%"
)

If row and column spacing are equal, the pixels are isotropic **inside the 2D slice**.

That does not automatically mean that the complete 3D volume is isotropic, because the spacing between slices can be different.

## 22. Link the DICOM metadata back to the CSV

In [ ]:
series_df = series_df.merge(
    train_series[
        [
            "StudyInstanceUID",
            "SeriesInstanceUID",
            "Anatomical_Plane",
            "Fluid_Sensitive",
            "Fat_Suppression"
        ]
    ],
    left_on=["study_uid", "series_uid"],
    right_on=["StudyInstanceUID", "SeriesInstanceUID"],
    how="left"
)

display(
    series_df[
        [
            "manufacturer",
            "magnetic_field_strength",
            "rows",
            "columns",
            "pixel_spacing_row",
            "pixel_spacing_col",
            "Anatomical_Plane",
            "Fluid_Sensitive",
            "Fat_Suppression"
        ]
    ].head()
)

The DICOM identifiers make it easy to connect the header information back to `train_series.csv`.

## Takeaways

- DICOM files contain much more than the MRI pixel values.
- We can inspect the acquisition metadata without loading the image itself.
- A study contains several series, and each series contains several DICOM slices.
- For acquisition metadata, series-level statistics are more meaningful than counting every slice.
- Manufacturer, field strength, matrix size, slice thickness and spatial resolution vary across the data.
- `PixelSpacing` gives the physical pixel size inside the image plane.
- Some DICOM tags are missing because not every field is mandatory.

### Next notebook

[**3. RSNA Knee -> See the MRI: From DICOM to Slices & Studies**](https://www.kaggle.com/code/h17ann/3-rsna-knee-see-the-mri-ax-cor-sag)

Next, we will load the pixel data and visualize complete MRI series.